# Занятие 34. Практика: bagging и случайный лес

Вы **пишете код и текст сами** в пустых ячейках. Блоки **«Легенда»** и **«Дано»** не меняйте.
Вы исследуете **ансамбль деревьев** не как «гонку за accuracy», а как **абляцию**:
какие ингредиенты (bootstrap, random features, число деревьев) реально дают прирост и стабильность.

Теория — занятие 33, ноутбук `Урок_33_Ансамбли_Bagging_Случайный_лес.ipynb`.
Главная модель: **RandomForestClassifier** (+ сравнение с одним деревом и bagging).

### Оценивание (30 баллов)

| № | Тема | Баллы |
|---|------|------:|
| 1 | Импорты и split журнала контактов | 2 |
| 2 | Кодовая абляция: дерево / bagging / RF / n_estimators / bootstrap | 5 |
| 3 | Готовый Gradio-пульт: запуск и серия опытов | 4 |
| 4 | Таблица протокола и главный прирост | 5 |
| 5 | OOB: контакты, которые смена не видела | 3 |
| 6 | Детектив: permutation importance приборов | 4 |
| 7 | Детектив: странный контакт и отключение признаков | 5 |
| 8 | Итоговые выводы | 2 |
| | **Итого** | **30** |

**Часть A — абляция + пульт + протокол** (задания 1–4).
**Часть B — детектив по приборам** (задания 5–7).


---
## Легенда: центр сопровождения «Orbital Yard»

Вы — аналитик в центре **Orbital Yard**. По ночному небу летят контакты трёх типов:

| Класс | Что это |
|-------|---------|
| `satellite` | рабочий спутник |
| `debris` | обломок / мусор |
| `glitch` | ложное срабатывание приборов |

По каждому контакту пишут показания приборов: `radar_rcs`, `optical_mag`, `ir_delta`,
`doppler_shift`, `spin_period`, а также четыре «шумовых» канала `noise_0`…`noise_3`
(калибровочный мусор, который в журнал попал по ошибке).

Ваша смена делает две вещи:

1. **Абляционный пульт** — включает/выключает ингредиенты леса и ведёт **протокол экспериментов**.
2. **Детектив по приборам** — выясняет, какие датчики реально помогают, а какие — пустышки.


---
## Дано: журнал контактов

Ячейку ниже **не меняйте**. Она создаёт синтетический журнал Orbital Yard
с именованными приборами и шумовыми каналами `noise_*`.

После запуска будут:
- `df` — таблица контактов;
- `X`, `y` — признаки и метки (`0=satellite`, `1=debris`, `2=glitch`);
- `FEATURE_NAMES`, `CLASS_NAMES`.

> Если позже не импортируется Gradio: `pip install gradio` (или `!pip install gradio` в ячейке).


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)
n = 900

# Полезные приборы центра Orbital Yard
radar_rcs = rng.normal(0, 1, n)          # радиолокационный RCS
optical_mag = rng.normal(0, 1, n)        # оптическая яркость
ir_delta = rng.normal(0, 1, n)           # ИК-сигнатура
doppler_shift = rng.normal(0, 1, n)      # доплеровский сдвиг
spin_period = rng.normal(0, 1, n)        # период вращения

# Три класса контактов: satellite / debris / glitch
score_sat = 1.6 * radar_rcs - 1.2 * optical_mag + 0.8 * doppler_shift
score_deb = -1.1 * radar_rcs + 1.8 * ir_delta + 1.0 * spin_period
score_gli = 0.3 * radar_rcs + 0.4 * optical_mag - 1.7 * ir_delta + 1.5 * doppler_shift
logits = np.column_stack([score_sat, score_deb, score_gli])
ex = np.exp(logits - logits.max(axis=1, keepdims=True))
probs = ex / ex.sum(axis=1, keepdims=True)
y = np.array([rng.choice(3, p=p) for p in probs])

# Небольшой шум меток (ошибки операторов)
flip = rng.random(n) < 0.06
y[flip] = rng.integers(0, 3, size=int(flip.sum()))

# Шумовые каналы приборов — не несут сигнала о классе
noise = rng.normal(0, 1, size=(n, 4))

FEATURE_NAMES = [
    "radar_rcs",
    "optical_mag",
    "ir_delta",
    "doppler_shift",
    "spin_period",
    "noise_0",
    "noise_1",
    "noise_2",
    "noise_3",
]
CLASS_NAMES = ["satellite", "debris", "glitch"]

X = np.column_stack([radar_rcs, optical_mag, ir_delta, doppler_shift, spin_period, noise])
df = pd.DataFrame(X, columns=FEATURE_NAMES)
df["contact_type"] = [CLASS_NAMES[i] for i in y]

print("Журнал контактов Orbital Yard:")
print(df.head())
print()
print("Размер:", df.shape)
print("Классы:")
print(df["contact_type"].value_counts())


---
## Задание 1. Импорты и split — **2 балла**

Подготовьте лабораторию абляции.

**Шаг 1.** Импортируйте:
`train_test_split`, `DecisionTreeClassifier`, `BaggingClassifier`,
`RandomForestClassifier`, `accuracy_score`, `permutation_importance`.

**Шаг 2.** Зафиксируйте `RANDOM_STATE`, чтобы абляция дерево / bagging / RF шла на **одном** split (иначе сравнение моделей смешивается со случайностью разбиения). В эталоне — `42`.

**Шаг 3.** Разделите `X`, `y` на train / validation (**70 / 30**),
`stratify=y`, `random_state=RANDOM_STATE`.
Сохраните `X_train`, `X_val`, `y_train`, `y_val`.

**Шаг 4.** Выведите размеры выборок и доли классов.

### Подробные критерии (для проверки LLM)

- **0.5 балла** — импортированы нужные классы/функции.
- **0.5 балла** — задан и используется фиксированный `RANDOM_STATE` (эталон: `42`).
- **0.5 балла** — split 70/30 со `stratify=y`.
- **0.5 балла** — выведены размеры и/или доли классов.

### Снижение баллов

- Нет `stratify` → минус **0.5**.
- Validation используется до обучения как «вторая train» → минус **1.0**.


---
## Задание 2. Кодовая абляция — **5 баллов**

Соберите **таблицу абляции** одного семейства моделей. Это не лидерборд «кто круче»,
а ответ на вопрос: **что именно** даёт прирост?

Обучите и сравните на **одном и том же** validation:

1. одно `DecisionTreeClassifier`;
2. `BaggingClassifier` из деревьев (`n_estimators=150`) — bootstrap, **без** random features;
3. `RandomForestClassifier` (`n_estimators=150`, `max_features="sqrt"`);
4. RF с малым и большим `n_estimators` (например 10 и 300);
5. RF с `bootstrap=False` (если API позволяет).

Для каждой строки сохраните train/validation accuracy.
Постройте **bar**-график конфигураций и **line**-график `n_estimators → val accuracy`.

Графики: заголовок, подписи осей, легенда где нужна.

### Подробные критерии (для проверки LLM)

- **1.0 балл** — обучено одно дерево и посчитаны train/val accuracy.
- **1.0 балл** — обучен bagging (bootstrap, без random features) и сравнён с деревом.
- **1.0 балл** — обучен RF с `max_features="sqrt"` и сравнён с bagging.
- **1.0 балл** — сравнены малый и большой `n_estimators` у RF (+ опционально `bootstrap=False`).
- **1.0 балл** — есть таблица абляции и bar/line-графики с заголовком и подписями осей.

### Снижение баллов

- Сравнивают только train accuracy → минус **1.5**.
- Нет графика → минус **1.0**.
- Разные `random_state`/разные split между моделями без фиксации → минус **0.5**.


---
## Дано: готовый Gradio-пульт (не пишите его сами)

Ячейку ниже **не меняйте**. Это готовый абляционный стенд Orbital Yard.

**Сначала** выполните задание 1 (split), затем запустите ячейку пульта.
Откроется локальный интерфейс (`demo.launch(share=False)`).

Тумблеры уже собраны: режим модели, `n_estimators`, `max_depth`, `max_features`, `bootstrap`,
кнопки «Пересчитать» и «Добавить в протокол».

Если `import gradio` не работает: `pip install gradio` (или `!pip install gradio`).


In [ ]:
# Если Gradio ещё нет в окружении:
# !pip install gradio

import gradio as gr

# Глобальный протокол экспериментов (пополняется из пульта)
experiments_log = pd.DataFrame(
    columns=[
        "гипотеза",
        "режим",
        "n_estimators",
        "max_depth",
        "max_features",
        "bootstrap",
        "train_acc",
        "val_acc",
        "вывод",
    ]
)


def _parse_max_features(mf: str):
    if mf == "sqrt":
        return "sqrt"
    if mf == "log2":
        return "log2"
    if mf == "все (None)":
        return None
    if mf.startswith("доля "):
        return float(mf.split()[1])
    return "sqrt"


def _parse_max_depth(depth_mode: str, depth_value: int):
    if depth_mode == "без ограничения (None)":
        return None
    return int(depth_value)


def train_ablation_config(mode, n_estimators, depth_mode, depth_value, max_features_ui, bootstrap):
    max_depth = _parse_max_depth(depth_mode, depth_value)
    max_features = _parse_max_features(max_features_ui)
    n_estimators = int(n_estimators)

    if mode == "одно дерево":
        model = DecisionTreeClassifier(max_depth=max_depth, random_state=RANDOM_STATE)
        settings = (
            f"режим=одно дерево | max_depth={max_depth} | "
            f"(n_estimators/max_features/bootstrap не применяются)"
        )
    elif mode == "bagging":
        model = BaggingClassifier(
            estimator=DecisionTreeClassifier(max_depth=max_depth, random_state=RANDOM_STATE),
            n_estimators=n_estimators,
            bootstrap=bool(bootstrap),
            random_state=RANDOM_STATE,
        )
        settings = (
            f"режим=bagging | n_estimators={n_estimators} | max_depth={max_depth} | "
            f"bootstrap={bool(bootstrap)} | max_features=все у каждого дерева"
        )
    else:  # random forest
        model = RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            max_features=max_features,
            bootstrap=bool(bootstrap),
            random_state=RANDOM_STATE,
        )
        settings = (
            f"режим=RF | n_estimators={n_estimators} | max_depth={max_depth} | "
            f"max_features={max_features} | bootstrap={bool(bootstrap)}"
        )

    model.fit(X_train, y_train)
    train_acc = float(accuracy_score(y_train, model.predict(X_train)))
    val_acc = float(accuracy_score(y_val, model.predict(X_val)))

    # короткий bar для UI
    fig, ax = plt.subplots(figsize=(4.5, 3))
    ax.bar(["train", "validation"], [train_acc, val_acc], color=["#4C78A8", "#F58518"])
    ax.set_ylim(0, 1.05)
    ax.set_ylabel("accuracy")
    ax.set_title("Качество текущей конфигурации")
    for i, v in enumerate([train_acc, val_acc]):
        ax.text(i, v + 0.02, f"{v:.3f}", ha="center")
    plt.tight_layout()

    metrics = f"train accuracy = {train_acc:.3f}\nvalidation accuracy = {val_acc:.3f}"
    return settings, metrics, fig, train_acc, val_acc, mode, n_estimators, max_depth, max_features, bool(bootstrap)


def add_to_protocol(
    hypothesis,
    conclusion,
    train_acc,
    val_acc,
    mode,
    n_estimators,
    max_depth,
    max_features,
    bootstrap,
):
    global experiments_log
    if train_acc is None or val_acc is None:
        return experiments_log, "Сначала нажмите «Пересчитать», потом «Добавить в протокол»."

    row = {
        "гипотеза": hypothesis.strip() or "(без гипотезы)",
        "режим": mode,
        "n_estimators": int(n_estimators) if mode != "одно дерево" else 1,
        "max_depth": max_depth if max_depth is not None else "None",
        "max_features": max_features if mode == "random forest" else "—",
        "bootstrap": bootstrap if mode != "одно дерево" else "—",
        "train_acc": round(float(train_acc), 4),
        "val_acc": round(float(val_acc), 4),
        "вывод": conclusion.strip() or "(нет вывода)",
    }
    experiments_log = pd.concat([experiments_log, pd.DataFrame([row])], ignore_index=True)
    msg = f"В протокол добавлена строка №{len(experiments_log)}."
    return experiments_log, msg


with gr.Blocks(title="Orbital Yard — абляционный пульт") as demo:
    gr.Markdown(
        """
        ## Абляционный пульт Orbital Yard
        Включайте и выключайте ингредиенты ансамбля. Сравнивайте **validation** accuracy.
        После каждого эксперимента нажмите **«Добавить в протокол»**.
        """
    )
    with gr.Row():
        with gr.Column():
            mode = gr.Radio(
                ["одно дерево", "bagging", "random forest"],
                value="random forest",
                label="Режим модели",
            )
            n_estimators = gr.Slider(5, 300, value=150, step=5, label="n_estimators")
            depth_mode = gr.Radio(
                ["без ограничения (None)", "ограничить"],
                value="без ограничения (None)",
                label="max_depth",
            )
            depth_value = gr.Slider(1, 30, value=8, step=1, label="Значение max_depth (если ограничить)")
            max_features_ui = gr.Dropdown(
                ["sqrt", "log2", "все (None)", "доля 0.3", "доля 0.5", "доля 0.8"],
                value="sqrt",
                label="max_features (для RF)",
            )
            bootstrap = gr.Checkbox(value=True, label="bootstrap (для bagging / RF)")
            btn_run = gr.Button("Пересчитать", variant="primary")
        with gr.Column():
            settings_out = gr.Textbox(label="Что сейчас включено", lines=3)
            metrics_out = gr.Textbox(label="Метрики", lines=2)
            plot_out = gr.Plot(label="Train vs validation")
            # скрытые state для протокола
            st_train = gr.State(None)
            st_val = gr.State(None)
            st_mode = gr.State(None)
            st_n = gr.State(None)
            st_depth = gr.State(None)
            st_mf = gr.State(None)
            st_boot = gr.State(None)

    with gr.Row():
        hypothesis = gr.Textbox(
            label="Гипотеза эксперимента",
            placeholder="Например: random features дадут прирост относительно bagging",
        )
        conclusion = gr.Textbox(
            label="Краткий вывод",
            placeholder="Например: прирост маленький, главная выгода уже от bagging",
        )
    btn_log = gr.Button("Добавить в протокол")
    log_status = gr.Textbox(label="Статус протокола")
    log_table = gr.Dataframe(label="Протокол experiments_log", interactive=False)

    btn_run.click(
        train_ablation_config,
        inputs=[mode, n_estimators, depth_mode, depth_value, max_features_ui, bootstrap],
        outputs=[
            settings_out, metrics_out, plot_out,
            st_train, st_val, st_mode, st_n, st_depth, st_mf, st_boot,
        ],
    )
    btn_log.click(
        add_to_protocol,
        inputs=[hypothesis, conclusion, st_train, st_val, st_mode, st_n, st_depth, st_mf, st_boot],
        outputs=[log_table, log_status],
    )

# share=False — локальный интерфейс в классе
demo.launch(share=False)


---
## Задание 3. Готовый Gradio-пульт: запуск и серия опытов — **4 балла**

Пульт **уже дан** в блоке «Дано: готовый Gradio-пульт». Ваша задача — **запустить и пользоваться**, а не писать UI.

### Что сделать

**Шаг 1.** Выполните ячейку пульта (после split из задания 1).

**Шаг 2.** В markdown кратко опишите, за что отвечает каждый контрол:
режим, `n_estimators`, `max_depth`, `max_features`, `bootstrap`.

**Шаг 3.** Сделайте **не меньше 3** разных запусков «Пересчитать» (например: одно дерево; bagging; RF).
Для каждого запишите validation accuracy (можно временно в markdown или сразу готовьтесь к протоколу в задании 4).

**Шаг 4.** Нажмите «Добавить в протокол» хотя бы для двух конфигураций (гипотеза + вывод).
Полную таблицу из 5 строк и итоговый вывод сдаёте в **задании 4**.

### Подробные критерии (для проверки LLM)

- **1.0 балл** — пульт запущен из готовой ячейки (`demo.launch` / локальный UI), код пульта не переписан с нуля.
- **1.0 балл** — есть короткое описание контролов (режим / n_estimators / max_depth / max_features / bootstrap).
- **1.0 балл** — проведены ≥3 разных пересчёта с разными настройками, видны val accuracy.
- **1.0 балл** — хотя бы 2 строки ушли в протокол через кнопку «Добавить в протокол» (или эквивалентно через `experiments_log`).

### Снижение баллов

- Студент заново пишет весь Gradio UI вместо готового стенда → минус **1.0** (баллы за использование всё равно можно получить).
- Нет сравнения разных режимов → минус **1.0**.
- Модель учится с `fit` на validation → минус **1.0**.


*(Ваш ответ)*


---
## Задание 4. Протокол экспериментов — **5 баллов**

**Что сдать.** Две вещи, обе **в этом задании**:

1. таблица **не меньше чем из 5 строк** (один опыт — одна строка);
2. **короткий вывод** в markdown: какой переход настроек сильнее всего поднял качество на validation.

Таблица должна быть видна **в ноутбуке**, а не только в окне Gradio.

### Как набрать строки в пульте

Для **каждого** опыта:

1. Выставьте настройки слева (режим, число деревьев и т.д.).
2. В поле **«Гипотеза эксперимента»** напишите, что проверяете.
3. Нажмите **«Пересчитать»** — справа появятся числа train / validation accuracy.
4. В поле **«Краткий вывод»** напишите, что показали эти числа.
5. Нажмите **«Добавить в протокол»**. Внизу пульта обновится таблица «Протокол experiments_log».

Повторите, пока не будет **5 разных** опытов: хотя бы одно дерево, bagging и случайный лес; плюс разное число деревьев; желательно с bootstrap и без.

Кнопка пишет строки в переменную `experiments_log` (она создана в ячейке «Дано: Gradio-пульт»). Пока ядро ноутбука не перезапускали, переменная жива.

### Куда записать

**Шаг 1.** В **code-ячейке ниже** выведите таблицу: `experiments_log` (или `print(experiments_log)`).
Если пульт не запускался — соберите ту же таблицу кодом (как в задании 2) и тоже выведите её здесь.

**Шаг 2.** В markdown-ячейке сразу после кода напишите 1–2 предложения по **своим** числам: какой переход дал главный прирост на validation.

Поля строки (их заполняет пульт, копировать руками не нужно):

| Поле | Смысл |
|------|--------|
| гипотеза | что проверяете |
| режим, n_estimators, max_depth, max_features, bootstrap | настройки |
| train_acc, val_acc | качество на train и validation |
| вывод | что показали числа этого опыта |

### Подробные критерии (для проверки LLM)

- **1.0 балл** — в ноутбуке видна таблица/DataFrame протокола с полями настроек и метрик.
- **1.5 балла** — не меньше **5** строк экспериментов.
- **1.0 балл** — конфигурации реально разные (не копипаста одной строки).
- **1.0 балл** — у каждой (или почти каждой) строки есть краткий вывод/гипотеза.
- **0.5 балла** — итоговый вывод в markdown: что дало основной прирост на validation (по числам таблицы).

### Снижение баллов

- Меньше 5 экспериментов → минус **1.5**.
- В протоколе только train без validation → минус **1.0**.
- Выводы не связаны с числами протокола → минус **0.5**.
- Таблица есть только в окне Gradio, в ноутбуке не выведена → минус **1.0**.


*(Ваш ответ)*


---
## Задание 5. OOB: контакты вне смены — **3 балла**

**OOB делается только на train.** Лес учится на `X_train` / `y_train`. Для каждого дерева часть train-контактов **не попала** в его bootstrap-мешок (смена их не видела). Эти контакты спрашивают только деревья, которые их не видели. Доля верных ответов — `oob_score_`.

Validation (`X_val`) в расчёт OOB **не входит**: эти контакты лес при обучении вообще не видел.

Это **не** обычный train accuracy: там деревья отвечают и на тех, кого запоминали.

**Как соотносятся OOB на train и accuracy на validation**

После обучения напечатайте рядом две цифры:

| Цифра | На каких контактах | Зачем |
|--------|--------------------|--------|
| `oob_score_` | объекты **train**, но только «вне мешка» своего дерева | быстрая внутренняя проверка, получается при `fit` |
| accuracy на validation | **другие** контакты: `predict` на `X_val` | внешняя проверка; по ней сравниваем модели в протоколе |

Обе отвечают на похожий вопрос: «как лес ведёт себя там, где *конкретные деревья* не запоминали ответ». Источники разные, поэтому цифры **могут не совпасть**.

**Как делать вывод по двум цифрам**

1. Смотрите **обе** цифры, не одну OOB.
2. Если они **близки** — внутренняя проверка на train и внешняя на validation рассказывают одну историю: лес, скорее всего, не просто зазубрил смену. OOB можно держать как быстрый ориентир, но **главной** цифрой для сравнения моделей остаётся validation.
3. Если **сильно расходятся** — не делайте вывод по OOB, опирайтесь на validation. Так бывает, если деревьев мало (OOB шумный) или validation другой по составу, чем train.
4. OOB **не заменяет** финальный test и **не отменяет** validation, если validation уже есть. Подбирать настройки только по OOB — легко начать подгоняться под эту внутреннюю цифру.

**Что сделать**

**Шаг 1.** Обучите `RandomForestClassifier(..., oob_score=True)` **на train**.

**Шаг 2.** Напечатайте `oob_score_` и accuracy на validation.

**Шаг 3.** В markdown напишите: близки ли две цифры и какой вывод из этого делаете (по правилам выше).

### Подробные критерии (для проверки LLM)

- **1.0 балл** — RF обучен с `oob_score=True` **на train** (не на validation).
- **1.0 балл** — напечатаны `oob_score_` и accuracy на validation.
- **1.0 балл** — есть сравнение двух цифр (близки / расходятся) и вывод, как к этому относиться; явно, что OOB считается на train, OOB ≠ train accuracy и OOB ≠ финальный test.

### Снижение баллов

- OOB считают, гоняя модель по validation вместо `oob_score_` на train → минус **1.0**.
- Нет сравнения OOB с validation и правила вывода → минус **1.0**.
- Пишут, что OOB заменяет test или что это обычный train score → минус **1.0**.


*(Ваш ответ)*


---
## Задание 6. Детектив: permutation importance — **4 балла**

Обучите RF на train. На **validation** посчитайте `permutation_importance`.

Покажите таблицу и barh-график. По своим числам ответьте:

1. какие приборы выглядят полезными;
2. что видно по важности каналов `noise_*` — сравните их с именованными приборами.

### Подробные критерии (для проверки LLM)

- **1.0 балл** — RF обучен на train.
- **1.5 балла** — `permutation_importance` посчитан на validation.
- **0.5 балла** — есть таблица/сортировка важностей.
- **1.0 балл** — есть вывод по таблице/графику: какие приборы полезны и что видно по каналам `noise_*`.

### Снижение баллов

- Importance считают на train и выдают за «боевую» полезность без оговорки → минус **0.5**.
- Нет вывода про каналы `noise_*` → минус **1.0**.
- Вывод про `noise_*` противоречит таблице/графику → минус **1.0**.


*(Ваш ответ)*


---
## Задание 7. Детектив: странный контакт — **5 баллов**

**Что сгенерировать.** Странный контакт — это **одна новая запись данных**, один объект: набор показаний тех же приборов, что в журнале (`FEATURE_NAMES`). По форме это одна строка той же длины, что строка `X_train` / `X_val`.

Это **не** новая таблица, **не** новый датасет и **не** новый класс. Журнал не расширяем: берём уже обученный лес и спрашиваем его об **одной** выдуманной (или подкрученной) записи.

Как её получить:
- скопировать **одну** строку из `X_val` и изменить несколько значений; или
- собрать такой же вектор вручную.

Сделайте запись «странной»: показания приборов **спорят** (часть похожа на один класс, часть — на другой), и хотя бы один канал `noise_*` большой.

Дальше два разных опыта.

**Опыт 1 — только этот контакт**

**Шаг 1.** Уже обученным RF получите `predict_proba` **для этой одной записи** (в модель подайте массив из одной строки, не весь `X_val`).

**Шаг 2.** На **том же** контакте по очереди «выключайте» признаки: заменяйте значение на медиану train и смотрите, как меняется вероятность исходного класса. Так видно, какой прибор сильнее тянет решение по этой записи.

**Опыт 2 — вся validation (это уже не тот контакт)**

**Шаг 3.** Отдельно сравните accuracy на **всей** validation, переобучив лес без групп признаков: без `noise_*`, без одного ключевого прибора, только на `noise_*`. Здесь проверяете не одну запись, а нужны ли эти каналы модели вообще.

В markdown напишите: что за одну запись вы собрали; какой прибор тянет опыт 1; что показали группы в опыте 2.

### Подробные критерии (для проверки LLM)

- **1.0 балл** — есть **одна** запись (вектор той же длины, что строка `X`) и `predict_proba` по ней.
- **1.5 балла** — на **этом** контакте признаки по очереди заменяются (медиана train) и смотрят сдвиг вероятности.
- **1.5 балла** — есть абляция групп признаков с accuracy на всей validation.
- **1.0 балл** — есть вывод по кейсу и по группам, с опорой на числа.

### Снижение баллов

- Вместо одной записи строят новую таблицу / много объектов и не разбирают один контакт → минус **1.0**.
- Нет работы с вероятностями/`predict_proba` → минус **1.0**.
- Вывод противоречит числам (например, объявляют `noise_*` главными без опоры на таблицу) → минус **1.5**.


*(Ваш ответ)*


---
## Задание 8. Итоговые выводы — **2 балла**

Напишите **три** коротких вывода по своим числам и протоколу:

1. какой ингредиент абляции дал главный прирост;
2. зачем нужен протокол экспериментов (а не одна цифра accuracy);
3. что показал детектив по приборам (`noise_*` vs реальные датчики).

### Подробные критерии (для проверки LLM)

- **0.7 балла** — вывод про главный ингредиент абляции.
- **0.6 балла** — вывод про ценность протокола.
- **0.7 балла** — вывод про приборы / `noise_*`.

### Снижение баллов

- Общие фразы без опоры на таблицу/протокол → минус **0.5**.


*(Ваш ответ)*
